In [19]:
import re
import csv
from pathlib import Path
from collections import defaultdict, Counter

In [20]:
CODE_DIR = 'dataset/storybook/src/code'
CODE_DIR_PATH = Path(CODE_DIR)

# 'components' or 'uis'
TYPE = 'uis'
TYPE_DIR_PATH = CODE_DIR_PATH / TYPE

COMPLEXITIES = ['simple', 'medium', 'hard']
VARIANTS = ['pretty', 'messy']

# Directory structure
# for components
#   {TYPE}/gt/{complexity}/{1-10}.vue
#   {TYPE}/a/{complexity}/{file}.vue
#   {TYPE}/{b|c|d}/{prompt_strategy}/{complexity}/{file}.vue
# for uis
#   {TYPE}/gt/{1-5}.vue                          <- FLAT, no complexity/variant subfolder
#   {TYPE}/a/{variant}/{1-5}-a.vue                <- variant, but NO prompt_strategy level
#   {TYPE}/{b|c|d}/{prompt_strategy}/{variant}/{file}.vue
APPROACHES = ['gt', 'a', 'b', 'c', 'd']

# Generic name for whichever second-level grouping applies for the current
# TYPE -- a complexity level for 'components', a pretty/messy variant for
# 'uis'. Used throughout instead of hardcoding COMPLEXITIES, so the same
# loading/aggregation code works for both directory layouts.
GROUP_LEVELS = COMPLEXITIES if TYPE == 'components' else VARIANTS
EXPECTED_GROUPS = set(GROUP_LEVELS)

# Approach 'a' is rule-based/deterministic and genuinely has NO prompt
# strategy -- not even a conceptual one, unlike B/C/D. It is still stored in
# the same 3-level {prompt_strategy}/{group}/{stem} structure as B/C/D below,
# but under the real value None (not an invented label like 'deterministic'),
# consistent with how storybook-generate-stories.ipynb already represents
# 'a' having no prompt_strategy. None works fine as a dict key here and
# avoids having to special-case 'a' anywhere downstream (aggregation, CSV
# export) that already iterates "whatever prompt_strategy keys exist".
A_PROMPT_STRATEGY_KEY = None

# Placeholder 'group' key for GT under TYPE == 'uis', which has no
# complexity/variant subfolder at all (GT is variant-independent, just like
# the Figma JSON/screenshot it comes from).
GT_FLAT_GROUP_KEY = '_flat'

PRIMEVUE_COMPONENTS = {
    'Accordion', 'AccordionPanel', 'AccordionHeader', 'AccordionContent',
    'Avatar', 'AvatarGroup',
    'Badge', 'Breadcrumb', 'Button',
    'Card', 'Checkbox', 'Column', 'ColumnGroup',
    'DataTable', 'DatePicker', 'Dialog', 'Divider',
    'InputNumber', 'InputText',
    'Menu',
    'OverlayBadge',
    'Password', 'Popover', 'ProgressBar',
    'RadioButton', 'Row',
    'Select', 'Skeleton', 'Slider',
    'Tab', 'TabList', 'TabPanel', 'TabPanels', 'Tabs', 'Tag', 'Textarea',
    'ToggleSwitch',
    'IconField', 'InputIcon',
}

PRIMEVUE_COMPONENTS_NORM = {c.lower(): c for c in PRIMEVUE_COMPONENTS}

SKIP_PROPS = {
    'class', 'style', 'id', 'ref', 'key',
    'v-model', 'v-if', 'v-else', 'v-for', 'v-show',
    'v-model:visible',
}

## Load reference and generated code files in groups by method, prompt strategy and complexity

In [21]:
CODE_FILES_BY_APPROACH: dict = {
    'gt': defaultdict(dict),
    'a':  defaultdict(lambda: defaultdict(dict)),
    'b':  defaultdict(lambda: defaultdict(dict)),
    'c':  defaultdict(lambda: defaultdict(dict)),
    'd':  defaultdict(lambda: defaultdict(dict)),
}

for approach in APPROACHES:
    approach_dir = TYPE_DIR_PATH / approach

    print(f'Loading code files for approach {approach} from {approach_dir}')

    if not approach_dir.exists():
        print(f'    Skip (not existing): {approach_dir}')

        continue

    if approach == 'gt':
        if TYPE == 'components':
            # gt/{complexity}/{file}.vue
            for complexity_dir in approach_dir.iterdir():
                if not complexity_dir.is_dir() or complexity_dir.name not in EXPECTED_GROUPS:
                    continue

                complexity = complexity_dir.name

                for vue_file in complexity_dir.glob('*.vue'):
                    content = vue_file.read_text(encoding='utf-8', errors='ignore')

                    if vue_file.stem in CODE_FILES_BY_APPROACH['gt'][complexity]:
                        print(f'  WARNING: Duplicate file {vue_file.stem} in {complexity} for gt, overwriting previous content.')

                    CODE_FILES_BY_APPROACH['gt'][complexity][vue_file.stem] = content

        else:
            # uis: gt/{1-5}.vue -- flat, GT is variant-independent (same file
            # used as reference for both pretty- and messy-derived generations).
            for vue_file in approach_dir.glob('*.vue'):
                content = vue_file.read_text(encoding='utf-8', errors='ignore')

                if vue_file.stem in CODE_FILES_BY_APPROACH['gt'][GT_FLAT_GROUP_KEY]:
                    print(f'  WARNING: Duplicate file {vue_file.stem} for gt, overwriting previous content.')

                CODE_FILES_BY_APPROACH['gt'][GT_FLAT_GROUP_KEY][vue_file.stem] = content

    elif approach == 'a':
        # components: a/{complexity}/{file}.vue
        # uis:        a/{variant}/{file}.vue
        # No prompt_strategy level either way -- stored under the fixed
        # A_PROMPT_STRATEGY_KEY placeholder so the rest of the notebook can
        # treat 'a' uniformly with b/c/d without special-casing it.
        for group_dir in approach_dir.iterdir():
            if not group_dir.is_dir() or group_dir.name not in EXPECTED_GROUPS:
                continue

            group = group_dir.name

            for vue_file in group_dir.glob('*.vue'):
                content = vue_file.read_text(encoding='utf-8', errors='ignore')

                if vue_file.stem in CODE_FILES_BY_APPROACH['a'][A_PROMPT_STRATEGY_KEY][group]:
                    print(f'  WARNING: Duplicate file {vue_file.stem} in {group} for a, overwriting previous content.')

                CODE_FILES_BY_APPROACH['a'][A_PROMPT_STRATEGY_KEY][group][vue_file.stem] = content

    else:
        # {approach}/{prompt_strategy}/{complexity_or_variant}/{file}.vue
        for strategy_dir in approach_dir.iterdir():

            if not strategy_dir.is_dir():
                continue

            prompt_strategy = strategy_dir.name

            for group_dir in strategy_dir.iterdir():
                if not group_dir.is_dir() or group_dir.name not in EXPECTED_GROUPS:
                    continue

                group = group_dir.name

                for vue_file in group_dir.glob('*.vue'):
                    content = vue_file.read_text(encoding='utf-8', errors='ignore')

                    if vue_file.stem in CODE_FILES_BY_APPROACH[approach][prompt_strategy][group]:
                        print(f'  WARNING: Duplicate file {vue_file.stem} in {group} for {approach}/{prompt_strategy}, overwriting previous content.')

                    CODE_FILES_BY_APPROACH[approach][prompt_strategy][group][vue_file.stem] = content

print('\nCode files loaded:')
for approach, data in CODE_FILES_BY_APPROACH.items():

    if approach == 'gt':
        total_files = sum(len(files) for files in data.values())

        print(f'Approach {approach}: {total_files} files')

    else:
        total_files = sum(len(files) for strategies in data.values() for files in strategies.values())

        print(f'Approach {approach}: {total_files} files')

        for strategy, groups in data.items():
            total_files = sum(len(files) for files in groups.values())
            strategy_label = strategy if strategy is not None else '(none -- deterministic, approach a)'

            print(f'  Strategy {strategy_label}: {total_files} files')

            for group, files in groups.items():
                print(f'    Group {group}: {len(files)} files')

Loading code files for approach gt from dataset\storybook\src\code\uis\gt
Loading code files for approach a from dataset\storybook\src\code\uis\a
Loading code files for approach b from dataset\storybook\src\code\uis\b
Loading code files for approach c from dataset\storybook\src\code\uis\c
Loading code files for approach d from dataset\storybook\src\code\uis\d

Code files loaded:
Approach gt: 5 files
Approach a: 10 files
  Strategy (none -- deterministic, approach a): 10 files
    Group messy: 5 files
    Group pretty: 5 files
Approach b: 180 files
  Strategy few_shot: 90 files
    Group messy: 45 files
    Group pretty: 45 files
  Strategy zero_shot: 90 files
    Group messy: 45 files
    Group pretty: 45 files
Approach c: 180 files
  Strategy few_shot: 90 files
    Group messy: 45 files
    Group pretty: 45 files
  Strategy zero_shot: 90 files
    Group messy: 45 files
    Group pretty: 45 files
Approach d: 180 files
  Strategy few_shot: 90 files
    Group messy: 45 files
    Group pr

## Component F1-Scores (P,R,F1)

In [22]:
def extract_template_block(sfc: str) -> str | None:
    """Finds the outer <template> block to be deeply robust against nested
    named-slot templates (<template #slotname>...</template>), which frequently occur in PrimeVue SFCs
    (icon slots, card headers, DataTable column bodies, ...).

    A non-greedy regex ‘<template>(.*?)</template>’ would stop at the FIRST
    closing </template>—that is, at a nested slot,
    not at the actual end of the SFC template—and thereby silently omit all
    subsequent components.
    """
    start = re.search(r'<template>', sfc)

    if not start:
        return None

    pos = start.end()
    depth = 1

    for m in re.finditer(r'<template(?:\s[^>]*)?(/?)>|</template>', sfc[pos:]):
        token = m.group(0)

        if token == '</template>':
            depth -= 1

            if depth == 0:
                return sfc[pos: pos + m.start()]
        elif not token.endswith('/>'):
            depth += 1   # nested, non-self-closing <template ...>

    return sfc[pos:]  # Fallback if no end is found (should not happen)


def extract_components_detailed(sfc: str) -> dict:
    """Breaks down all tags in the <template> block into three categories:

    - recognized:               Multiset (list) of known PrimeVue components,
                                canonically normalized as with `extract_components()` previously.
    - unrecognized_components:  Counter of PascalCase tags that look LIKE a PrimeVue
                                component but are not listed in PRIMEVUE_COMPONENTS.
                                Two possible causes: (a) the model
                                is “hallucinating” a component name, or (b) the
                                PRIMEVUE_COMPONENTS catalog is incomplete.
    - native_elements:          Counter of lowercase (native HTML) tags.
                                As expected, these are common (div, span, label, ...),
                                but they serve as the raw basis for a later fallback rate.
    """
    template = extract_template_block(sfc)

    recognized: list[str] = []
    unrecognized_components: Counter = Counter()
    native_elements: Counter = Counter()

    if template is None:
        return {
            'recognized': recognized,
            'unrecognized_components': unrecognized_components,
            'native_elements': native_elements,
        }

    for m in re.finditer(r'<([A-Za-z][A-Za-z0-9]*)', template):
        tag = m.group(1)
        canonical = PRIMEVUE_COMPONENTS_NORM.get(tag.lower())

        if canonical:
            recognized.append(canonical)
        elif tag[0].isupper():
            unrecognized_components[tag] += 1
        else:
            native_elements[tag] += 1

    return {
        'recognized': recognized,
        'unrecognized_components': unrecognized_components,
        'native_elements': native_elements,
    }

In [23]:
def compute_f1(generated: list[str], ground_truth: list[str]) -> dict:
    """Precision/Recall/F1 via a multiset comparison of component tags.

    A simple set comparison would incorrectly count multiple instances of the same type as a single hit.
    """
    g = Counter(generated)
    r = Counter(ground_truth)

    if not g and not r:
        return {
            'precision': 1.0, 'recall': 1.0, 'f1': 1.0,
            'tp': 0, 'gen_count': 0, 'gt_count': 0,
            'false_positives': Counter(), 'false_negatives': Counter(),
        }

    tp = sum(min(g[c], r[c]) for c in set(g) | set(r))

    gen_count = sum(g.values())
    gt_count  = sum(r.values())

    precision = tp / gen_count if gen_count else 0.0
    recall    = tp / gt_count  if gt_count  else 0.0

    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) > 0 else 0.0
    )

    fp = Counter({c: max(g[c] - r[c], 0) for c in g if g[c] > r[c]})
    fn = Counter({c: max(r[c] - g[c], 0) for c in r if r[c] > g[c]})

    return {
        'precision': round(precision, 4),
        'recall':    round(recall, 4),
        'f1':        round(f1, 4),
        'tp':        tp,
        'gen_count': gen_count,
        'gt_count':  gt_count,
        'false_positives': fp,
        'false_negatives': fn,
    }

In [24]:
def parse_generated_stem(stem: str, approach: str) -> dict | None:
    """Parses filenames such as:
      ‘1-a’                          -> Approach A (deterministic, no model/run)
      '1-b2-claude-sonnet-5-1'       -> Approach B, Strategy b2, Model, Run

    Returns: {‘index’, ‘strategy’, ‘model’, ‘run’} or None if the pattern
    does not match (the approach must already be known—it comes from the directory level).
    """
    parts = stem.split('-')

    if not parts[0].isdigit():
        return None

    index = parts[0].zfill(2)

    if approach == 'a':
        if len(parts) == 2 and parts[1] == 'a':
            return {'index': index, 'strategy': 'a', 'model': None, 'run': None}

        return None

    if len(parts) >= 4 and re.match(rf'^{approach}[123]$', parts[1]) and parts[-1].isdigit():
        strategy = parts[1]
        model = '-'.join(parts[2:-1])
        run = parts[-1]

        return {'index': index, 'strategy': strategy, 'model': model, 'run': run}

    return None


def gt_index(stem: str) -> str:
    """Normalizes a GT stem (‘1’, ‘01’, ...) to the 2-digit index."""
    return stem.zfill(2) if stem.isdigit() else stem


def format_counter(c: Counter) -> str:
    """'Button:2;Card:1' for CSV storage; leave blank instead of ‘{}’ if the counter is empty."""
    return ';'.join(f'{k}:{v}' for k, v in sorted(c.items()))

In [25]:
f1_results: list[dict] = []

for group in GROUP_LEVELS:
    # GT is variant-independent for 'uis' (flat, single set of files shared by
    # both pretty and messy) but genuinely per-complexity for 'components' --
    # look it up accordingly rather than assuming a per-group GT set always exists.
    gt_group_key = GT_FLAT_GROUP_KEY if TYPE == 'uis' else group
    gt_files = CODE_FILES_BY_APPROACH['gt'].get(gt_group_key, {})
    gt_by_index = {gt_index(stem): content for stem, content in gt_files.items()}
    gt_detail_by_index = {idx: extract_components_detailed(sfc) for idx, sfc in gt_by_index.items()}

    for approach in ['a', 'b', 'c', 'd']:
        approach_data = CODE_FILES_BY_APPROACH[approach]

        for prompt_strategy, by_group in approach_data.items():
            gen_files = by_group.get(group, {})

            for stem, gen_sfc in gen_files.items():
                parsed = parse_generated_stem(stem, approach)

                if parsed is None:
                    print(f'   WARNING: Filename does not match expected pattern: {approach}/{prompt_strategy}/{group}/{stem}')

                    continue

                gt_detail = gt_detail_by_index.get(parsed['index'])

                if gt_detail is None:
                    print(f'  WARNING: No GT found for {approach}/{prompt_strategy}/{group}/{stem} (Index {parsed["index"]})')

                    continue

                gen_detail = extract_components_detailed(gen_sfc)

                gt_comps  = gt_detail['recognized']
                gen_comps = gen_detail['recognized']

                scores = compute_f1(gen_comps, gt_comps)

                unrecognized_n = sum(gen_detail['unrecognized_components'].values())
                native_n       = sum(gen_detail['native_elements'].values())

                print(f'{group}/{parsed["index"]} '
                      f'[{approach}/{prompt_strategy or "-"}/{parsed["strategy"]}/{parsed["model"] or "-"}]  '
                      f'P={scores["precision"]:.2f}  R={scores["recall"]:.2f}  F1={scores["f1"]:.2f}'
                      + (f'  unrecognized={unrecognized_n}' if unrecognized_n else '')
                      + (f'  native={native_n}' if native_n else ''))

                f1_results.append({
                    'mockup':          f'{group}-{parsed["index"]}',
                    'complexity':       group,  # complexity for 'components', variant for 'uis'
                    'index':            parsed['index'],
                    'approach':         approach,
                    'strategy':         parsed['strategy'],
                    'prompt_strategy':  prompt_strategy,
                    'model':            parsed['model'] or '',
                    'run':              parsed['run'] or '',
                    'precision':        scores['precision'],
                    'recall':           scores['recall'],
                    'f1':               scores['f1'],
                    'tp':               scores['tp'],
                    'gt_count':         scores['gt_count'],
                    'gen_count':        scores['gen_count'],
                    'fp_types':         format_counter(scores['false_positives']),
                    'fn_types':         format_counter(scores['false_negatives']),
                    'unrecognized_n':      unrecognized_n,
                    'unrecognized_types':  format_counter(gen_detail['unrecognized_components']),
                    'native_element_n':    native_n,
                    'native_element_types': format_counter(gen_detail['native_elements']),
                })

print(f'\nComputed: {len(f1_results)} F1-Scores')

pretty/01 [a/-/a/-]  P=0.68  R=0.96  F1=0.79  native=37
pretty/02 [a/-/a/-]  P=0.94  R=0.79  F1=0.86  native=13
pretty/03 [a/-/a/-]  P=0.84  R=0.80  F1=0.82  native=21
pretty/04 [a/-/a/-]  P=0.86  R=0.90  F1=0.88  native=26
pretty/05 [a/-/a/-]  P=0.84  R=0.94  F1=0.89  native=46
pretty/01 [b/few_shot/b1/claude-sonnet-5]  P=0.89  R=0.96  F1=0.93  native=25
pretty/01 [b/few_shot/b1/gemini-3.1-pro-preview]  P=0.83  R=0.96  F1=0.89  native=38
pretty/01 [b/few_shot/b1/gpt-5.6-terra]  P=0.75  R=0.92  F1=0.83  unrecognized=1  native=36
pretty/01 [b/few_shot/b2/claude-sonnet-5]  P=0.80  R=0.92  F1=0.86  native=28
pretty/01 [b/few_shot/b2/gemini-3.1-pro-preview]  P=0.71  R=0.96  F1=0.82  native=39
pretty/01 [b/few_shot/b2/gpt-5.6-terra]  P=0.81  R=0.96  F1=0.88  native=33
pretty/01 [b/few_shot/b3/claude-sonnet-5]  P=0.80  R=0.92  F1=0.86  native=29
pretty/01 [b/few_shot/b3/gemini-3.1-pro-preview]  P=0.78  R=0.96  F1=0.86  native=37
pretty/01 [b/few_shot/b3/gpt-5.6-terra]  P=0.75  R=0.92  F1=0.8

In [26]:
def aggregate_macro(items: list[dict]) -> dict:
    n = len(items)
    return {
        'precision_macro': round(sum(i['precision'] for i in items) / n, 4),
        'recall_macro':    round(sum(i['recall']    for i in items) / n, 4),
        'f1_macro':        round(sum(i['f1']        for i in items) / n, 4),
        'n': n,
    }


def aggregate_micro(items: list[dict]) -> dict:
    tp  = sum(i['tp'] for i in items)
    gen = sum(i['gen_count'] for i in items)
    gt  = sum(i['gt_count']  for i in items)

    precision = tp / gen if gen else 0.0
    recall    = tp / gt  if gt  else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

    return {
        'precision_micro': round(precision, 4),
        'recall_micro':    round(recall, 4),
        'f1_micro':        round(f1, 4),
        'n': len(items),
    }


def group_key(r: dict) -> tuple:
    """Grouping by approach, strategy, model, AND prompt strategy ->
    Zero-Shot and Few-Shot must not be included in the same mean, since both
    are different independent variables in the design."""
    return (r['approach'], r['strategy'], r['model'], r['prompt_strategy'])


def group_label(key: tuple) -> str:
    approach, strategy, model, prompt_strategy = key
    base = strategy if not model else f'{strategy}_{model}'

    # prompt_strategy is genuinely None for approach 'a' (no artificial
    # placeholder label) -- omit the suffix entirely instead of rendering
    # Python's None as the literal string 'None' in printed output.
    return f'{base}_{prompt_strategy}' if prompt_strategy else base


by_method: dict[tuple, list[dict]] = defaultdict(list)
for r in f1_results:
    by_method[group_key(r)].append(r)

by_method_complexity: dict[tuple, list[dict]] = defaultdict(list)
for r in f1_results:
    by_method_complexity[(group_key(r), r['complexity'])].append(r)

GROUPS = sorted(by_method.keys(), key=lambda k: (k[0], k[1], k[2] or '', k[3]))

print(f'\n{"Method":42s} {"P(macro)":>9s} {"R(macro)":>9s} {"F1(macro)":>9s} {"F1(micro)":>9s} {"n":>4s}')
print('-' * 88)

for key in GROUPS:
    items = by_method[key]
    macro = aggregate_macro(items)
    micro = aggregate_micro(items)

    print(f'{group_label(key):42s} {macro["precision_macro"]:9.3f} {macro["recall_macro"]:9.3f} '
          f'{macro["f1_macro"]:9.3f} {micro["f1_micro"]:9.3f} {macro["n"]:4d}')


Method                                      P(macro)  R(macro) F1(macro) F1(micro)    n
----------------------------------------------------------------------------------------
a                                              0.415     0.439     0.424     0.570   10
b1_claude-sonnet-5_few_shot                    0.942     0.859     0.894     0.897   10
b1_claude-sonnet-5_zero_shot                   0.925     0.792     0.844     0.844   10
b1_gemini-3.1-pro-preview_few_shot             0.930     0.858     0.889     0.885   10
b1_gemini-3.1-pro-preview_zero_shot            0.886     0.755     0.800     0.808   10
b1_gpt-5.6-terra_few_shot                      0.865     0.882     0.870     0.865   10
b1_gpt-5.6-terra_zero_shot                     0.871     0.820     0.840     0.835   10
b2_claude-sonnet-5_few_shot                    0.953     0.856     0.895     0.892   10
b2_claude-sonnet-5_zero_shot                   0.895     0.825     0.848     0.844   10
b2_gemini-3.1-pro-preview_few_

In [27]:
EVALUATIONS_DIR = Path(f'evaluations/{TYPE}')
EVALUATIONS_DIR.mkdir(parents=True, exist_ok=True)

# 1) Long format: one row per mockup x configuration
BY_MOCKUP_CSV = EVALUATIONS_DIR / f'eval_f1_{TYPE}_by_mockup.csv'
BY_MOCKUP_FIELDNAMES = [
    'mockup', 'complexity', 'index', 'approach', 'strategy', 'prompt_strategy', 'model', 'run',
    'precision', 'recall', 'f1', 'tp', 'gt_count', 'gen_count', 'fp_types', 'fn_types',
    'unrecognized_n', 'unrecognized_types', 'native_element_n', 'native_element_types',
]

with open(BY_MOCKUP_CSV, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=BY_MOCKUP_FIELDNAMES)
    writer.writeheader()
    writer.writerows(f1_results)

print(f'Saved: {BY_MOCKUP_CSV}  ({len(f1_results)} rows)')

# 2) Summary: Macro- and Micro-Aggregation by Approach/Strategy/Model/Prompt Strategy
SUMMARY_CSV = EVALUATIONS_DIR / f'eval_f1_{TYPE}_summary.csv'
SUMMARY_FIELDNAMES = [
    'approach', 'strategy', 'prompt_strategy', 'model',
    'precision_macro', 'recall_macro', 'f1_macro',
    'precision_micro', 'recall_micro', 'f1_micro', 'n',
    'avg_unrecognized_n', 'avg_native_element_n',
]

summary_rows = []
for key in GROUPS:
    approach, strategy, model, prompt_strategy = key
    items = by_method[key]

    macro = aggregate_macro(items)
    micro = aggregate_micro(items)

    avg_unrecognized = round(sum(i['unrecognized_n'] for i in items) / len(items), 4)
    avg_native       = round(sum(i['native_element_n'] for i in items) / len(items), 4)

    summary_rows.append({
        'approach':        approach,
        'strategy':        strategy,
        'prompt_strategy': prompt_strategy,
        'model':           model or '',
        **macro,
        **{k: v for k, v in micro.items() if k != 'n'},
        'avg_unrecognized_n':   avg_unrecognized,
        'avg_native_element_n': avg_native,
    })

with open(SUMMARY_CSV, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=SUMMARY_FIELDNAMES)
    writer.writeheader()
    writer.writerows(summary_rows)

print(f'Saved: {SUMMARY_CSV}  ({len(summary_rows)} rows)')

# 3) Per configuration x complexity level (basis for UF4 degradation factor / UF5)
BY_COMPLEXITY_CSV = EVALUATIONS_DIR / f'eval_f1_{TYPE}_by_complexity.csv'
BY_COMPLEXITY_FIELDNAMES = [
    'approach', 'strategy', 'prompt_strategy', 'model', 'complexity',
    'precision_macro', 'recall_macro', 'f1_macro',
    'precision_micro', 'recall_micro', 'f1_micro', 'n',
]

by_complexity_rows = []
for key in GROUPS:
    approach, strategy, model, prompt_strategy = key

    for group in GROUP_LEVELS:
        items = by_method_complexity[(key, group)]

        if not items:
            continue

        macro = aggregate_macro(items)
        micro = aggregate_micro(items)

        by_complexity_rows.append({
            'approach':        approach,
            'strategy':        strategy,
            'prompt_strategy': prompt_strategy,
            'model':           model or '',
            'complexity':      group,  # complexity for 'components', variant for 'uis'
            **macro,
            **{k: v for k, v in micro.items() if k != 'n'},
        })

with open(BY_COMPLEXITY_CSV, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=BY_COMPLEXITY_FIELDNAMES)
    writer.writeheader()
    writer.writerows(by_complexity_rows)

print(f'Saved: {BY_COMPLEXITY_CSV}  ({len(by_complexity_rows)} rows)')

# 4a) Degradation factor (medium/simple, hard/simple) per configuration (UF4)
#     -- 'components' only, since 'simple'/'medium'/'hard' is an ordered
#     complexity progression that does not apply to 'uis'.
def degradation_factor(f1_base: float | None, f1_target: float | None) -> float | None:
    if f1_base is None or f1_target is None or f1_base == 0:
        return None
    return round(f1_target / f1_base, 4)


if TYPE == 'components':
    DEGRADATION_CSV = EVALUATIONS_DIR / f'eval_f1_{TYPE}_degradation.csv'
    DEGRADATION_FIELDNAMES = [
        'approach', 'strategy', 'prompt_strategy', 'model',
        'f1_simple', 'f1_medium', 'f1_hard',
        'degradation_factor_medium', 'degradation_factor_hard',
    ]

    degradation_rows = []
    for key in GROUPS:
        approach, strategy, model, prompt_strategy = key

        simple_items = by_method_complexity[(key, 'simple')]
        medium_items = by_method_complexity[(key, 'medium')]
        hard_items   = by_method_complexity[(key, 'hard')]

        simple_f1 = aggregate_macro(simple_items)['f1_macro'] if simple_items else None
        medium_f1 = aggregate_macro(medium_items)['f1_macro'] if medium_items else None
        hard_f1   = aggregate_macro(hard_items)['f1_macro']   if hard_items   else None

        degradation_rows.append({
            'approach':        approach,
            'strategy':        strategy,
            'prompt_strategy': prompt_strategy,
            'model':           model or '',
            'f1_simple':       simple_f1,
            'f1_medium':       medium_f1,
            'f1_hard':         hard_f1,
            'degradation_factor_medium': degradation_factor(simple_f1, medium_f1),
            'degradation_factor_hard':   degradation_factor(simple_f1, hard_f1),
        })

    with open(DEGRADATION_CSV, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=DEGRADATION_FIELDNAMES)
        writer.writeheader()
        writer.writerows(degradation_rows)

    print(f'Saved: {DEGRADATION_CSV}  ({len(degradation_rows)} rows)')

else:
    # 4b) Pretty/Messy robustness ratio per configuration -- this is the UF5
    #     Robustheitsindex R(M,P) = F1_messy / F1_pretty, the 'uis' equivalent
    #     of the complexity degradation factor above (input-quality
    #     degradation instead of input-complexity increase).
    ROBUSTNESS_CSV = EVALUATIONS_DIR / f'eval_f1_{TYPE}_robustness.csv'
    ROBUSTNESS_FIELDNAMES = [
        'approach', 'strategy', 'prompt_strategy', 'model',
        'f1_pretty', 'f1_messy', 'robustness_index',
    ]

    robustness_rows = []
    for key in GROUPS:
        approach, strategy, model, prompt_strategy = key

        pretty_items = by_method_complexity[(key, 'pretty')]
        messy_items  = by_method_complexity[(key, 'messy')]

        pretty_f1 = aggregate_macro(pretty_items)['f1_macro'] if pretty_items else None
        messy_f1  = aggregate_macro(messy_items)['f1_macro']  if messy_items  else None

        robustness_rows.append({
            'approach':        approach,
            'strategy':        strategy,
            'prompt_strategy': prompt_strategy,
            'model':           model or '',
            'f1_pretty':       pretty_f1,
            'f1_messy':        messy_f1,
            'robustness_index': degradation_factor(pretty_f1, messy_f1),
        })

    with open(ROBUSTNESS_CSV, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=ROBUSTNESS_FIELDNAMES)
        writer.writeheader()
        writer.writerows(robustness_rows)

    print(f'Saved: {ROBUSTNESS_CSV}  ({len(robustness_rows)} rows)')

Saved: evaluations\uis\eval_f1_uis_by_mockup.csv  (550 rows)
Saved: evaluations\uis\eval_f1_uis_summary.csv  (55 rows)
Saved: evaluations\uis\eval_f1_uis_by_complexity.csv  (110 rows)
Saved: evaluations\uis\eval_f1_uis_robustness.csv  (55 rows)
